# Office hours: pulling a BigQuery table into the evaluation template

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/team-simmer/ai4tm/blob/main/week_3/office_hours_load_from_bigquery.ipynb)

The [CSV and Google Sheet walkthrough](office_hours_load_csv_and_evaluate.ipynb) gets your labeled data into a pandas table called `raw`, then has OpenRouter label it and runs `evaluate()`. This notebook swaps the first part for BigQuery: if your Search Console data (or anything else you want to evaluate) already lives in a BigQuery table, here's how to get it from there into the same `raw` table, and nothing more. From that point on, Steps 2 to 7 of the CSV walkthrough work unchanged (same OpenRouter client and default model).

The example table is the same Larkspur Trail Co. Search Console sample as the CSV walkthrough, loaded into BigQuery: `ai-4-tm.examples.search_console_topics`.

**Time**: ~15 minutes
**Cost**: Free for a table this size. BigQuery gives every account 1 TiB of queries a month at no charge, and Step 4 checks how much a query will scan before it runs. Labeling with OpenRouter is a fraction of a cent for a table this size.

> **How to run this notebook.** Open it in Cursor, VS Code, or Jupyter on your own computer. Create this week's environment once (`cd` into the `week_3` folder, then `uv venv` and `uv pip install -r requirements.txt`), select that `.venv` as the kernel, and keep your OpenRouter key in the project `.env` file. The setup cell also installs the BigQuery client. Full walkthrough: [Week 2's setup guide](../week_2/lesson08_setup_guide.ipynb).

## Before you start: two permissions, in two places

Reading a BigQuery table from code takes two separate permissions, and missing either one gives an `Access Denied` that doesn't say which one is missing:

| Role | Granted on | What it lets you do |
|---|---|---|
| **BigQuery Data Viewer** | the dataset (or table) | read the data and its schema |
| **BigQuery Job User** | a project, usually your own | run a query |

(There's an optional third, *BigQuery Read Session User*, for faster downloads of large results. This notebook doesn't need it, see Step 4.)

The second one surprises people. Every query is a *job*, and a job runs in, and is billed to, a project. It doesn't have to be the project that owns the table: you can read a table in `ai-4-tm` while the query runs in your own project. That's why Step 2 asks for two things, the table and a *billing project*.

If you weren't given access to the course table, the optional cell in Step 2 copies the sample CSV into a table in your own project, so you can follow along there instead.

**Privacy reminder from Week 4**: a warehouse table usually holds a lot more than you need. Select only the columns you're evaluating (Step 4 does), and check for personal data before anything leaves BigQuery: the next step sends every row to OpenRouter. See the [PII protection notebook](../week_4/lesson17_pii_protection.ipynb).

## Where to run this

- **Locally / VS Code / Cursor** (recommended for this course): use the `week_3` virtual environment, your project `.env` with `OPENROUTER_API_KEY`, and sign in once from a terminal with the Google Cloud CLI (Step 2 has the command).
- **Colab** (optional): sign in with a Google popup in Step 2. Add `OPENROUTER_API_KEY` to Colab Secrets (same name as in `.env`).
- **BigQuery Studio notebooks** (the notebook editor inside the BigQuery console): you're already signed in for BigQuery, but OpenRouter still needs an API key from `.env` or Secret Manager — the course default is local OpenRouter, not Vertex AI.

> **Staying in BigQuery Studio for the whole evaluation?** Loading the table works the same there. For labeling, you'll need `OPENROUTER_API_KEY` available to Python (Secret Manager or a manual setup cell), and the same OpenRouter + OpenAI library pattern as the CSV walkthrough.

## Setup

If you completed the Week 2 setup guide, your OpenRouter key is already in the project `.env` file as `OPENROUTER_API_KEY`. If you haven't, do that first: open `lesson08_setup_guide.ipynb` in the `week_2` folder.

**Locally**: use the same `week_3/.venv` environment as the [evaluation template](lesson12_evaluation_template.ipynb). Run the install cell below once, then the OpenRouter connection cell.

`google-cloud-bigquery` is Google's Python client for BigQuery, and `db-dtypes` is what it needs to turn query results into a pandas table. Colab ships both; locally they're installed on top of the `week_3` environment in the install cell.

In [1]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q openai python-dotenv pandas google-cloud-bigquery db-dtypes scikit-learn matplotlib seaborn tiktoken
else:
    import pathlib

    _cwd = pathlib.Path.cwd()
    if _cwd.name == "week_3" and (_cwd / "requirements.txt").exists():
        _req_path = "requirements.txt"
    elif (_cwd / "week_3" / "requirements.txt").exists():
        _req_path = "week_3/requirements.txt"
    elif (_cwd.parent / "week_3" / "requirements.txt").exists():
        _req_path = "../week_3/requirements.txt"
    else:
        _req_path = None

    if _req_path is None:
        print(
            "Couldn't find week_3/requirements.txt from the current working directory:",
            _cwd,
        )
        print("Set up your local environment first -- see Week 2's setup guide for the uv commands.")
    else:
        %pip install -q -r {_req_path}

    %pip install -q google-cloud-bigquery db-dtypes

Note: you may need to restart the kernel to use updated packages.


d:\Personal project\ai4tm\.venv\Scripts\python.exe: No module named pip


Note: you may need to restart the kernel to use updated packages.


d:\Personal project\ai4tm\.venv\Scripts\python.exe: No module named pip


In [2]:
from dotenv import load_dotenv
import os
from pathlib import Path
from openai import OpenAI

for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / ".env").exists():
        load_dotenv(_candidate / ".env")
        break
else:
    load_dotenv()

if IN_COLAB:
    from google.colab import userdata

    _api_key = userdata.get("OPENROUTER_API_KEY")
else:
    _api_key = os.getenv("OPENROUTER_API_KEY")

if not _api_key:
    raise ValueError(
        "OPENROUTER_API_KEY is missing. Add it to a .env file in the project folder "
        "(see week_2/lesson08_setup_guide.ipynb). In Colab, save it as OPENROUTER_API_KEY in Secrets."
    )

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=_api_key,
)

MODEL_NAME = "z-ai/glm-5.3-flash"  # change this line to switch models
print("Connected. Model:", MODEL_NAME)

Connected. Model: z-ai/glm-5.3-flash


## Step 1: sign in

The BigQuery client never takes a password or key in the code. It looks for Google credentials on the machine it runs on (*Application Default Credentials*), so this step puts yours there.

- **In Colab**: the cell below opens a Google sign-in popup. Pick the account that was given access to the table.
- **Locally**: the cell does nothing. Instead, run this once in a terminal (it opens a browser to sign in, and remembers it afterwards):

  ```
  gcloud auth application-default login
  ```

  No `gcloud`? Install the [Google Cloud CLI](https://cloud.google.com/sdk/docs/install) first. If you get `invalid_grant` or `Reauthentication failed` later, your saved login has expired or belongs to another account: run the command again.
- **In BigQuery Studio**: you're already signed in. Running the cell is harmless.

In [ ]:
if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Signed in.")
else:
    print("Not in Colab: using the login from `gcloud auth application-default login`.")

## Step 2: point at the table, and at a project to run the query in

- `TABLE_ID` is the full name of the table: `project.dataset.table`. In the BigQuery console, open the table and copy it from the **Details** tab (**Table ID**).
- `BILLING_PROJECT` is the project you have *BigQuery Job User* on, where the query runs. It's the project ID (like `my-project-123456`), not its display name; the project picker at the top of the Cloud console shows both.

The cell then looks the table up and prints its columns. That's a metadata request, not a query: it's free, and it only needs *Data Viewer*. If this works but Step 4 fails with `Access Denied`, the missing piece is *Job User* on `BILLING_PROJECT`.

In [ ]:
from google.cloud import bigquery

TABLE_ID = "ai-4-tm.examples.search_console_topics"
BILLING_PROJECT = ""      # your project ID, e.g. "my-project-123456"

if not BILLING_PROJECT:
    raise ValueError("Set BILLING_PROJECT to the ID of a project you can run queries in.")

bq_client = bigquery.Client(project=BILLING_PROJECT)

table = bq_client.get_table(TABLE_ID)
print(f"{table.full_table_id}: {table.num_rows:,} rows, {table.num_bytes:,} bytes")
for field in table.schema:
    print(f"  {field.name:<15} {field.field_type}")

### Optional: your own copy of the sample table

Only if you weren't given access to the course table. This loads the sample CSV from the [CSV walkthrough](office_hours_load_csv_and_evaluate.ipynb) (run its Step 0, or put `office_hours_search_console_topics.csv` next to this notebook) into a new table in `BILLING_PROJECT`, and points `TABLE_ID` at it. You'll need *BigQuery Data Editor* in that project, which you'll usually have in a project you own.

BigQuery is stricter about column names than a spreadsheet, so the headers are turned into lower-case names without spaces on the way in (`Top queries` becomes `top_queries`), and CTR is stored as a number rather than text. That's the same layout as the course table, so the rest of the notebook works on either.

In [ ]:
CREATE_OWN_COPY = False      # set to True to create the table

if CREATE_OWN_COPY:
    import pandas as pd

    DATASET = f"{BILLING_PROJECT}.ai4tm_examples"
    bq_client.create_dataset(DATASET, exists_ok=True)

    sample = pd.read_csv("office_hours_search_console_topics.csv").dropna(how="all")
    sample.columns = [c.strip().lower().replace(" ", "_") for c in sample.columns]
    # Same types as the course table: CTR as a fraction (0.0978), not text ("9.78%")
    sample["ctr"] = sample["ctr"].str.rstrip("%").astype(float) / 100

    TABLE_ID = f"{DATASET}.search_console_topics"
    job = bq_client.load_table_from_dataframe(
        sample, TABLE_ID,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
    )
    job.result()   # waits for the load to finish
    table = bq_client.get_table(TABLE_ID)
    print(f"Created {TABLE_ID} with {table.num_rows} rows. Columns: {[f.name for f in table.schema]}")
else:
    print("Skipped: using", TABLE_ID)

## Step 3: write the query

Pick the columns you need and rename them on the way out, so the result already has the names the rest of the walkthrough expects: `text` for what's being labeled, `true_label` for the topic a person assigned. Everything else, like impressions, is kept with its own name.

A few habits worth getting into, because on a real Search Console export (millions of rows, years of dates) they're the difference between a free query and a costly one:

- **Name the columns, don't `SELECT *`.** BigQuery stores each column separately and charges for the columns a query reads, so fewer columns means less data scanned.
- **`LIMIT` doesn't make a query cheaper.** It cuts the result, not what's read: `SELECT * ... LIMIT 10` on a 1 TB table scans 1 TB. To evaluate on a sample, filter with `WHERE` (a date range, say) instead.
- **Filter out rows with no label here**, rather than downloading them and dropping them later.
- **Add an `ORDER BY`** if row order matters to you. A table has no built-in order, so without one the rows can come back in a different order every run. Sorting doesn't change what's scanned.

In [ ]:
SQL = f"""
SELECT
    top_queries AS text,
    topic AS true_label,
    clicks,
    ctr,
    position,
    impressions
FROM `{TABLE_ID}`
WHERE topic IS NOT NULL
ORDER BY impressions DESC
"""
print(SQL)

## Step 4: check what the query will cost, then run it

The habit from [Week 2's token cost guide](../week_2/lesson09_token_cost_guide.ipynb), applied to BigQuery: a *dry run* asks BigQuery how much data the query would scan, without running it or charging anything.

On-demand queries cost about $6.25 per TiB scanned (check [current pricing](https://cloud.google.com/bigquery/pricing#on_demand_pricing)), the first 1 TiB each month is free, and every query is billed for at least 10 MB per table it reads. So on a small table like this one, the dry run is mostly there to make the habit stick.

In [ ]:
dry_run = bq_client.query(SQL, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
scanned = dry_run.total_bytes_processed
cost = max(scanned, 10 * 1024**2) / 1024**4 * 6.25

print(f"This query will scan {scanned / 1e6:.3f} MB, about ${cost:.6f} at on-demand prices (free within the monthly 1 TiB).")

Looks fine? Run it. `.to_dataframe()` downloads the result into a pandas table: the same `raw` table that every loading option in the CSV walkthrough ends with.

`create_bqstorage_client=False` is there on purpose. Colab has a faster download route installed (the BigQuery Storage API), and the client uses it automatically, but it needs one more permission, *BigQuery Read Session User*, which *Job User* doesn't include. Without it, the download fails with `403 ... bigquery.readsessions.create`, even though the query itself ran. For a table you're going to label row by row with OpenRouter (hundreds or thousands of rows, not millions), the normal route is plenty fast. If you're pulling a lot more and have the role, remove that argument.

In [ ]:
raw = bq_client.query(SQL).to_dataframe(create_bqstorage_client=False)

print(f"{len(raw)} rows, columns: {list(raw.columns)}")
raw.head()

## Step 5: check it looks like the template expects

The data came out of a database instead of a spreadsheet, so it's tidier than a CSV in some ways and different in others:

- **Numbers are already numbers.** BigQuery keeps column types, so impressions arrive as integers, with no `"1,400"` strings to clean up. Whole numbers show up as `Int64` below, pandas' integer type that allows empty values; it behaves like a normal number.
- **Empty values are `None` / `<NA>`**, not empty strings. The `WHERE ... IS NOT NULL` in the query already dropped rows without a label.
- **Labels are exactly as typed.** A database doesn't clean anything for you: `"Returns"` and `"returns "` are still two different values. Step 4 of the CSV walkthrough (`clean_labels()`) still matters, and the count below shows whether you'll need it.

In [ ]:
print(raw.dtypes, "\n")

# Each spelling of each label, and how many rows use it. Two lines that look like the
# same topic (different case, a stray space) are what clean_labels() fixes later.
print(raw["true_label"].map(repr).value_counts().sort_index().to_string())

## Step 6: carry on with the evaluation

`raw` now has the same shape as after Step 1 of the [CSV walkthrough](office_hours_load_csv_and_evaluate.ipynb), with one difference: the query already renamed the columns. So in its Step 2, use:

```python
TEXT_COLUMN = "text"
TRUE_COLUMN = "true_label"
PRED_COLUMN = "Model topic"
```

and in its Step 6, `WEIGHT_COLUMN = "impressions"`. Everything else (OpenRouter labeling with the same `client` and `MODEL_NAME` from Setup above, `clean_labels()`, `evaluate()`, reading the mismatches) works unchanged.

Every Colab notebook is a separate session, so `raw` doesn't travel to another notebook by itself. Two ways to continue:

1. **Same notebook** (simplest): copy the cells from Step 2 onwards of the CSV walkthrough and paste them below this one, then run them.
2. **Through a file**: run the cell below to save `raw` as a CSV, then load it with Option A of the CSV walkthrough. In Colab the file is downloaded to your computer, and you'll upload it into the other notebook's session.

In [ ]:
OUTPUT_PATH = "office_hours_from_bigquery.csv"

raw.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {OUTPUT_PATH}")

if IN_COLAB:
    from google.colab import files
    files.download(OUTPUT_PATH)

### Optional: writing the results back to BigQuery

Once the evaluation has run, the model's topics often belong next to the source data, for a dashboard, or to join back to the full Search Console table. After Step 7 of the CSV walkthrough you have a table called `out`; the line below writes it to a new BigQuery table (needs *Data Editor* on the destination dataset):

```python
bq_client.load_table_from_dataframe(out, f"{BILLING_PROJECT}.your_dataset.search_console_topics_evaluated").result()
```

Column names with spaces, like `Model topic`, are rejected by older BigQuery setups: rename them first if the load fails on that (`out.rename(columns={"Model topic": "model_topic"})`).

## Common errors

| Error | Usual cause | Fix |
|---|---|---|
| `DefaultCredentialsError` (locally) | Not signed in | Run `gcloud auth application-default login` in a terminal |
| `RefreshError: invalid_grant` / `Reauthentication failed` | Saved login expired, or belongs to an account that no longer exists | Run `gcloud auth application-default login` again with the right account |
| `403 Access Denied: ... bigquery.jobs.create` | No *BigQuery Job User* on `BILLING_PROJECT` | Ask for the role, or use a project you own |
| `403 Access Denied: Table ...` / `User does not have permission to query table` | No *BigQuery Data Viewer* on the table's dataset | Ask whoever owns the dataset to share it with your account |
| `404 Not found: Table` | Typo in `TABLE_ID`, or wrong project/dataset | Copy the Table ID from the table's **Details** tab |
| `400 Unrecognized name: ...` | A column in the `SELECT` doesn't exist in the table | Compare with the column list printed in Step 2 |
| `Project ... has not enabled BigQuery` / `BigQuery API has not been used in project` | `BILLING_PROJECT` doesn't have the BigQuery API turned on | Enable it in the Cloud console (**APIs & Services**), or pick another project |
| `403 ... bigquery.readsessions.create` when downloading | The faster Storage API download needs *BigQuery Read Session User* | Keep `create_bqstorage_client=False` in Step 4, or ask for the role |
| `ValueError: Please install the 'db-dtypes' package` | Missing package | Run the Setup cell again |
| Cost estimate far bigger than expected | `SELECT *`, or no `WHERE` on a large table | Name only the columns you need; filter by date |
| `ValueError` / missing key in Setup | `OPENROUTER_API_KEY` not in `.env` (local) or Colab Secrets | Follow Week 2's setup guide |